In [ ]:
import asyncio
import aiohttp

class VCPInstance:
    """Represents a single Charging Station + EV pair."""
    def __init__(self, base_url, instance_id, session: aiohttp.ClientSession):
        self.instance_id = instance_id
        self.session = session
        self.cp_url = f"{base_url}/{instance_id}/cp"
        self.ev_url = f"{base_url}/{instance_id}/ev"

    # --- CP (Charging Point) Methods ---
    async def cp_check_health(self):
        async with self.session.get(f"{self.cp_url}/") as r:
            return await r.json()

    async def cp_plugin(self, evse_id: str, connector_id: str, halfway: bool = True):
        params = {"evse_id": evse_id, "connector_id": connector_id, "halfway": str(halfway).lower()}
        async with self.session.post(f"{self.cp_url}/plugin", params=params) as r:
            return r.status

    async def cp_plugout(self, evse_id: str, connector_id: str):
        params = {"evse_id": evse_id, "connector_id": connector_id}
        async with self.session.post(f"{self.cp_url}/plugout", params=params) as r:
            return r.status

    async def cp_authorize(self, auth_id: str, id_type: str = None, evse_id: str = None, connector_id: str = None):
        params = {"id": auth_id, "type": id_type, "evse_id": evse_id, "connector_id": connector_id}
        params = {k: v for k, v in params.items() if v is not None}

        async with self.session.post(f"{self.cp_url}/authorize", params=params) as r:
            # content_type=None disables the strict check that caused your crash
            if "application/json" in r.headers.get("Content-Type", ""):
                return await r.json()
            else:
                return await r.text()

    async def cp_set_parking_bay(self, occupied: bool):
        async with self.session.post(f"{self.cp_url}/parkingbay", params={"occupied": str(occupied).lower()}) as r:
            return r.status

    async def cp_reboot(self):
        async with self.session.post(f"{self.cp_url}/reboot") as r:
            return r.status

    async def cp_set_state(self, connector_id: str, faulted: bool, unlock_failed: bool, evse_id: str = "1", error_type: str = "MREC6UnderVoltage"):
        params = {"connector_id": connector_id, "faulted": str(faulted).lower(), "unlock_failed": str(unlock_failed).lower(), "evse_id": evse_id, "error_type": error_type}
        async with self.session.post(f"{self.cp_url}/state", params=params) as r:
            return r.status

    # --- EV (Electric Vehicle) Methods ---
    async def ev_plugin(self, connector_id: str = "1", evse_id: str = None, soc: float = None):
        params = {"connector_id": connector_id, "evse_id": evse_id, "soc": soc}
        params = {k: v for k, v in params.items() if v is not None}
        async with self.session.post(f"{self.ev_url}/plugin", params=params) as r:
            return r.status

    async def ev_plugout(self, connector_id: str = "1", evse_id: str = None):
        params = {"connector_id": connector_id, "evse_id": evse_id}
        params = {k: v for k, v in params.items() if v is not None}
        async with self.session.post(f"{self.ev_url}/plugout", params=params) as r:
            return r.status

    async def ev_stop_charging(self):
        async with self.session.post(f"{self.ev_url}/end") as r:
            return r.status

    async def ev_set_ready(self, ready: bool):
        async with self.session.post(f"{self.ev_url}/state", params={"ready": str(ready).lower()}) as r:
            return r.status

class VirtualChargingPark:
    """Manages a collection of VCPInstances."""
    def __init__(self, base_url: str, instance_ids: list, session: aiohttp.ClientSession):
        self.instances = [VCPInstance(base_url, i_id, session) for i_id in instance_ids]

    async def run_parallel(self, coro_func):
        """Helper to run a specific function across all instances in the park."""
        return await asyncio.gather(*(coro_func(inst) for inst in self.instances))

Click the button to copy all instance ids in the cloud and paste them

In [ ]:
# @title 🔧 Configure your Park
BASE_URL = "https://vcp.pionix.com"
RAW_IDS = "" # @param {type:"string"}

# If the user left RAW_IDS empty, ask via input
if not RAW_IDS:
    print("Please paste your Instance IDs (comma separated). Press Ctrl+D (or Cmd+D) when finished:")
    import sys
    RAW_IDS = sys.stdin.read()

INSTANCE_IDS = [line.strip() for line in RAW_IDS.split(', ') if line.strip()]

print(f"✅ Park initialized with {len(INSTANCE_IDS)} instances.")

✅ Park initialized with 14 instances.


In [ ]:
async def simple_charging_session(instance: VCPInstance):
    prefix = f"[{instance.instance_id[:6]}]"

    print(f"{prefix} 🚗 Occupying parking bay...")
    await instance.cp_set_parking_bay(occupied=True)

    print(f"{prefix} 🔑 Authorizing...")
    await instance.cp_authorize(auth_id="97CDAC45")

    print(f"{prefix} 🔌 Plugging in (Both sides)...")
    await instance.cp_plugin(evse_id="1", connector_id="1")
    await instance.ev_plugin(connector_id="1")

    print(f"{prefix} ✅ Charging...")
    await asyncio.sleep(360)

    print(f"{prefix} 🏁 Finished.")
    await instance.cp_plugout(evse_id="1", connector_id="1")
    await instance.cp_set_parking_bay(occupied=False)

async def main():
    # Execute on the whole park
    async with aiohttp.ClientSession() as session:
        park = VirtualChargingPark(BASE_URL, INSTANCE_IDS, session)
        await park.run_parallel(simple_charging_session)

await main()

[9slnwq] 🚗 Occupying parking bay...
[h3m4ku] 🚗 Occupying parking bay...
[m3dw4u] 🚗 Occupying parking bay...
[ay0psc] 🚗 Occupying parking bay...
[vo09be] 🚗 Occupying parking bay...
[rhjwv7] 🚗 Occupying parking bay...
[ty1ph8] 🚗 Occupying parking bay...
[pyd74c] 🚗 Occupying parking bay...
[cfes3r] 🚗 Occupying parking bay...
[ark4yt] 🚗 Occupying parking bay...
[7g708i] 🚗 Occupying parking bay...
[v7nolr] 🚗 Occupying parking bay...
[x02c17] 🚗 Occupying parking bay...
[hempct] 🚗 Occupying parking bay...
[9slnwq] 🔑 Authorizing...
[cfes3r] 🔑 Authorizing...
[hempct] 🔑 Authorizing...
[ay0psc] 🔑 Authorizing...
[ty1ph8] 🔑 Authorizing...
[ark4yt] 🔑 Authorizing...
[pyd74c] 🔑 Authorizing...
[h3m4ku] 🔑 Authorizing...
[m3dw4u] 🔑 Authorizing...
[vo09be] 🔑 Authorizing...
[rhjwv7] 🔑 Authorizing...
[x02c17] 🔑 Authorizing...
[7g708i] 🔑 Authorizing...
[v7nolr] 🔑 Authorizing...
[9slnwq] 🔌 Plugging in (Both sides)...
[hempct] 🔌 Plugging in (Both sides)...
[ark4yt] 🔌 Plugging in (Both sides)...
[pyd74c] 🔌 Plug